# Практика 21 · Зміщення проти дисперсії

> 📖 **Теорія:** відкрий `lecture.html` у цій же теці.
> 📝 **Домашнє завдання:** `homework.md`. 🧠 **Тест:** `quiz.html`.

Лекція каже: помилка розкладається на три доданки — шум, зміщення² і дисперсію,
і цей розклад **точний**. Тут ми його порахуємо руками й переконаємось, що доданки
справді сумуються в загальну помилку.

**Що зробимо:**
1. Створимо «сорок паралельних світів» — багато навчальних вибірок з одного джерела
2. Побудуємо **віяло** прогнозів і побачимо дисперсію очима
3. Порахуємо bias², variance і шум **числом** та перевіримо, що вони складаються в
   очікувану помилку
4. Побудуємо криві train/test від складності — те, що бачить практик з однією вибіркою
5. Зробимо те саме через **бутстреп**, коли вибірка одна-єдина
6. Покажемо, що дані б'ють лише по дисперсії, і то як $1/n$
7. Обміняємо дисперсію на зміщення регуляризацією — і виграємо в сумі
8. Побудуємо те саме готовими інструментами `learning_curve` і `validation_curve`

## 0. Істина, шум і вибірка

Влаштовуємо світ, у якому ми **знаємо істину**. У житті так не буває — саме тому
bias і variance неможливо поміряти на реальних даних. Тут можна, бо істину
задали ми самі.

$$y = f(x) + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0,\ \sigma^2)$$

Функція $f$ навмисно **не поліном**: жоден скінченний степінь не відтворить її
точно, тому зміщення буде справжнім, а не штучним.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial import legendre

NOISE_SIGMA = 0.3          # стандартне відхилення шуму
SAMPLE_SIZE = 40           # скільки точок у кожній навчальній вибірці


def true_function(x):
    """Істина, якої модель не знає. Синусоїда — не поліном, і в цьому вся сіль."""
    return np.sin(1.7 * np.pi * x) + 0.6 * x ** 2


# сітка, на якій міряємо помилку; краї трохи обрізані — там будь-який поліном шаліє
x_test = np.linspace(-0.85, 0.85, 171)
f_test = true_function(x_test)


def draw_training_sample(rng, size=SAMPLE_SIZE):
    """Одна навчальна вибірка: випадкові точки плюс шум."""
    x = rng.uniform(-1, 1, size)
    y = true_function(x) + rng.normal(0, NOISE_SIGMA, size)
    return x, y


demo_rng = np.random.default_rng(1)
x_demo, y_demo = draw_training_sample(demo_rng)

print(f"розмір вибірки: {SAMPLE_SIZE} точок")
print(f"шум: σ = {NOISE_SIGMA}, отже незвідна частина помилки σ² = {NOISE_SIGMA ** 2:.4f}")
print(f"нижче цієї межі не опуститься жодна модель — навіть ідеальна")

## 1. Модель: поліном у базисі Лежандра

Складність моделі регулюємо степенем полінома. Але є технічна пастка, і про неї
варто знати.

Якщо будувати матрицю ознак із сирих степенів $[1, x, x^2, \dots, x^{12}]$, її
стовпці стають майже однаковими (усі виглядають як «щось, що росте»), матриця
погано обумовлена, і на високих степенях результат псується **чисельно**, а не
через перенавчання. Ми б міряли похибку арифметики замість дисперсії моделі.

**Поліноми Лежандра** дають той самий простір функцій, але їхні стовпці
майже ортогональні. Простір той самий — отже, прогноз МНК має збігтися до
останньої цифри. Це ми зараз і перевіримо.

In [ ]:
def fit_polynomial(x_train, y_train, degree, x_new):
    """МНК-поліном заданого степеня в базисі Лежандра. Повертає прогноз у точках x_new."""
    design = legendre.legvander(x_train, degree)          # стовпці: P₀(x), P₁(x), …, P_d(x)
    coefficients, *_ = np.linalg.lstsq(design, y_train, rcond=None)
    return legendre.legvander(x_new, degree) @ coefficients


our_prediction = fit_polynomial(x_demo, y_demo, 7, x_test)
print(f"прогноз у перших трьох точках: {np.round(our_prediction[:3], 6)}")

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

# та сама задача, але через звичайні степені й sklearn
library_model = make_pipeline(PolynomialFeatures(7), LinearRegression())
library_model.fit(x_demo.reshape(-1, 1), y_demo)
library_prediction = library_model.predict(x_test.reshape(-1, 1))

print(f"найбільше розходження: {np.abs(our_prediction - library_prediction).max():.2e}")

assert np.allclose(our_prediction, library_prediction), "розрахунок розійшовся!"
print("\n✅ збігається — базис Лежандра дає той самий простір функцій,")
print("   але поводиться чисельно набагато краще на високих степенях")

## 2. Сорок паралельних світів

Ось головна ідея розділу 03 лекції: зміщення й дисперсія — властивості **процедури
навчання**, а не однієї навченої моделі. Побачити їх на одному прогоні неможливо.

Тому влаштовуємо уявний експеримент по-справжньому: витягуємо багато різних
навчальних вибірок з того самого джерела, на кожній навчаємо свою модель і
дивимось на розкид прогнозів.

In [ ]:
N_WORLDS = 300     # скільки паралельних світів проживаємо


def run_parallel_worlds(degree, size=SAMPLE_SIZE, worlds=N_WORLDS, seed=0):
    """Навчає `worlds` моделей на різних вибірках. Повертає матрицю прогнозів.

    Рядок — один світ, стовпець — одна тестова точка.
    """
    rng = np.random.default_rng(seed)
    predictions = np.zeros((worlds, len(x_test)))
    for world in range(worlds):
        x_train, y_train = draw_training_sample(rng, size)
        predictions[world] = fit_polynomial(x_train, y_train, degree, x_test)
    return predictions


predictions_degree_7 = run_parallel_worlds(degree=7)
print(f"матриця прогнозів: {predictions_degree_7.shape} (світів × тестових точок)")
print(f"у точці x = 0 прогнози гуляють від {predictions_degree_7[:, 85].min():.2f} "
      f"до {predictions_degree_7[:, 85].max():.2f}")
print(f"істина там: {f_test[85]:.2f}")

### Віяло

Кожна тонка лінія — модель з окремого світу. Товста рожева — **середня модель**
$\bar{f}$, та сама, що стоїть у визначенні зміщення. Сірий пунктир — істина.

Дивись на дві речі **окремо**:
- наскільки товста лінія розходиться з пунктиром — це **зміщення**;
- наскільки широке віяло — це **дисперсія**.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)

for ax, degree in zip(axes, [1, 5, 12]):
    predictions = run_parallel_worlds(degree)
    # малюємо лише 40 світів — інакше картинка перетвориться на суцільну пляму
    for world in range(40):
        ax.plot(x_test, predictions[world], color="teal", alpha=.16, lw=1)
    ax.plot(x_test, predictions.mean(axis=0), color="crimson", lw=3, label="середня модель")
    ax.plot(x_test, f_test, color="grey", ls="--", lw=2, label="істина")
    ax.set_title(f"степінь {degree}")
    ax.set_xlabel("x"); ax.grid(alpha=.25)

axes[0].set_ylabel("прогноз"); axes[0].set_ylim(-2.5, 2.5); axes[0].legend(loc="upper left")
plt.tight_layout(); plt.show()

print("Степінь 1: усі прямі лежать майже одна на одній (дисперсії немає),")
print("           але жодна навіть не намагається повторити форму істини.")
print("Степінь 12: середня модель лягла на істину чудово, зате окрема модель")
print("           може дати будь-що. Це і є компроміс, буквально очима.")

## 3. Рахуємо три доданки числом

Формули з лекції, слово в слово:

$$\text{Зміщення}(x) = \mathbb{E}_D[\hat{f}_D(x)] - f(x), \qquad
\text{Дисперсія}(x) = \mathbb{E}_D\big[(\hat{f}_D(x) - \mathbb{E}_D[\hat{f}_D(x)])^2\big]$$

Сподівання $\mathbb{E}_D$ береться **по навчальних вибірках** — тобто по рядках
нашої матриці прогнозів. Потім усереднюємо по тестових точках, щоб отримати
одне число на модель.

In [ ]:
def decompose(predictions):
    """Розкладає помилку на три доданки. Повертає (зміщення², дисперсія, шум)."""
    average_model = predictions.mean(axis=0)                 # f̄(x): середнє по світах
    bias_squared = np.mean((average_model - f_test) ** 2)    # промах середньої моделі
    variance = np.mean(predictions.var(axis=0))              # розліт навколо середньої
    noise = NOISE_SIGMA ** 2                                 # підлога, однакова завжди
    return bias_squared, variance, noise


print(f"{'степінь':>8} {'зміщення²':>11} {'дисперсія':>11} {'шум':>8} {'разом':>9}")
for degree in [1, 5, 12]:
    bias_squared, variance, noise = decompose(run_parallel_worlds(degree))
    print(f"{degree:>8} {bias_squared:>11.4f} {variance:>11.4f} {noise:>8.4f} "
          f"{bias_squared + variance + noise:>9.4f}")

### Перевірка 1: доданки справді сумуються

Лекція наполягає, що розклад — **тотожність**, а не наближення. Перевіримо це
буквально: порахуємо середній квадрат відхилення прогнозів від істини напряму,
без жодного розкладу, і порівняємо зі сумою зміщення² та дисперсії.

In [ ]:
predictions = run_parallel_worlds(degree=7)
bias_squared, variance, noise = decompose(predictions)

# «в лоб»: середній по світах і по точках квадрат відхилення прогнозу від істини
straight_error = np.mean((predictions - f_test) ** 2)

print(f"зміщення² + дисперсія = {bias_squared + variance:.12f}")
print(f"пораховано напряму    = {straight_error:.12f}")

assert np.allclose(bias_squared + variance, straight_error), "розклад не сходиться!"
print("\n✅ тотожність підтверджена: доданків рівно два плюс шум, і вони не перекриваються")

### Перевірка 2: а тепер із справжнім шумом на тесті

Попередня перевірка була алгебраїчною: істину ми знали точно. Тепер зробимо
чесніше — у кожному світі згенеруємо **нові зашумлені** тестові відповіді, як
у реальному житті, і поміряємо помилку на них.

Тут уже працює статистика, тому точного збігу не буде — лише збіг у межах
похибки Монте-Карло. Саме так і має бути.

In [ ]:
noise_rng = np.random.default_rng(123)

# у кожному світі — свій свіжий шум на тестових точках
noisy_targets = f_test + noise_rng.normal(0, NOISE_SIGMA, predictions.shape)
measured_error = np.mean((noisy_targets - predictions) ** 2)

predicted_by_decomposition = bias_squared + variance + noise

print(f"поміряна помилка на зашумлених відповідях: {measured_error:.5f}")
print(f"передбачення розкладу (bias² + var + σ²):  {predicted_by_decomposition:.5f}")
print(f"розбіжність: {abs(measured_error - predicted_by_decomposition) / predicted_by_decomposition * 100:.2f}%")

assert np.allclose(measured_error, predicted_by_decomposition, rtol=0.05), \
    "розклад не описує реальну помилку!"
print("\n✅ розклад передбачає реальну помилку з точністю до похибки Монте-Карло")

## 4. Розклад по складності — головна картинка теми

Ті самі числа, але для всіх степенів одразу. Сірий фундамент однаковий скрізь:
це шум, підлога, нижче якої не опуститься ніхто.

In [ ]:
degrees = np.arange(1, 13)
bias_by_degree = np.zeros(len(degrees))
variance_by_degree = np.zeros(len(degrees))

for i, degree in enumerate(degrees):
    bias_by_degree[i], variance_by_degree[i], _ = decompose(run_parallel_worlds(degree))

total_by_degree = bias_by_degree + variance_by_degree + NOISE_SIGMA ** 2
best_degree = degrees[total_by_degree.argmin()]

print(f"{'степінь':>8} {'зміщення²':>11} {'дисперсія':>11} {'разом':>9}")
for i, degree in enumerate(degrees):
    mark = "  ← мінімум" if degree == best_degree else ""
    print(f"{degree:>8} {bias_by_degree[i]:>11.4f} {variance_by_degree[i]:>11.4f} "
          f"{total_by_degree[i]:>9.4f}{mark}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

noise_row = np.full(len(degrees), NOISE_SIGMA ** 2)
axes[0].bar(degrees, noise_row, color="lightgrey", label="шум σ²")
axes[0].bar(degrees, bias_by_degree, bottom=noise_row, color="crimson", label="зміщення²")
axes[0].bar(degrees, variance_by_degree, bottom=noise_row + bias_by_degree,
            color="teal", label="дисперсія")
axes[0].set_yscale("log")
axes[0].set_title("З чого складається помилка (лог. шкала)")

axes[1].plot(degrees, bias_by_degree, "o-", color="crimson", lw=2, label="зміщення²")
axes[1].plot(degrees, variance_by_degree, "o-", color="teal", lw=2, label="дисперсія")
axes[1].plot(degrees, total_by_degree, "o-", color="black", lw=2.5, label="разом")
axes[1].axvline(best_degree, ls="--", color="grey", label=f"мінімум суми: {best_degree}")
axes[1].set_yscale("log")
axes[1].set_title("Зміщення падає, дисперсія росте")

for ax in axes:
    ax.set_xlabel("степінь полінома"); ax.legend(); ax.grid(alpha=.25)
plt.tight_layout(); plt.show()

print(f"Мінімум суми — на степені {best_degree}. Зверни увагу: це не там, де")
print("найменше зміщення, і не там, де найменша дисперсія. Оптимізувати треба суму.")

## 5. Що бачить практик: криві train і test

Паралельних світів у житті немає — є одна вибірка. Що з неї видно?

Дві криві: помилка на навчальних даних і помилка на відкладених. Поводяться вони
принципово по-різному, і саме **розрив між ними** ставить діагноз.

In [ ]:
practice_rng = np.random.default_rng(2024)
x_train_one, y_train_one = draw_training_sample(practice_rng, size=SAMPLE_SIZE)

# відкладена вибірка з того самого джерела — те, що в житті називають тестом
x_holdout, y_holdout = draw_training_sample(practice_rng, size=400)

train_error = np.zeros(len(degrees))
holdout_error = np.zeros(len(degrees))

for i, degree in enumerate(degrees):
    prediction_on_train = fit_polynomial(x_train_one, y_train_one, degree, x_train_one)
    prediction_on_holdout = fit_polynomial(x_train_one, y_train_one, degree, x_holdout)
    train_error[i] = np.mean((y_train_one - prediction_on_train) ** 2)
    holdout_error[i] = np.mean((y_holdout - prediction_on_holdout) ** 2)

print(f"{'степінь':>8} {'train':>9} {'test':>9} {'розрив':>9}")
for i, degree in enumerate(degrees):
    print(f"{degree:>8} {train_error[i]:>9.4f} {holdout_error[i]:>9.4f} "
          f"{holdout_error[i] - train_error[i]:>9.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.4))

ax.plot(degrees, train_error, "o-", color="crimson", lw=2.5, label="train")
ax.plot(degrees, holdout_error, "o-", color="teal", lw=2.5, label="test (відкладені)")
ax.axhline(NOISE_SIGMA ** 2, ls=":", color="grey", label="підлога σ² — нижче не буває")
ax.set_yscale("log")
ax.set_xlabel("степінь полінома"); ax.set_ylabel("MSE (лог. шкала)")
ax.set_title("Навчальна помилка падає завжди — тому вона нічого не каже")
ax.legend(); ax.grid(alpha=.25)
plt.tight_layout(); plt.show()

print("Діагностика за п'ять хвилин:")
print("  обидві криві високі й поруч → зміщення → ускладнювати")
print("  розрив великий              → дисперсія → спрощувати або добувати дані")

## 6. Бутстреп: те саме, коли вибірка одна

Паралельні світи — уявний експеримент. Але дещо схоже можна зробити й насправді:
**бутстреп**. Беремо одну наявну вибірку й багато разів витягуємо з неї
$n$ обʼєктів **з поверненням**. Кожна така підвибірка трохи інша — от і
«паралельні світи», зроблені з підручних матеріалів.

Бутстреп не знає істини, тому зміщення він оцінити не може. А от **дисперсію** —
цілком: вона не потребує знання $f$.

In [ ]:
def bootstrap_predictions(x_train, y_train, degree, resamples=N_WORLDS, seed=5):
    """Багато моделей, навчених на підвибірках з поверненням з однієї вибірки."""
    rng = np.random.default_rng(seed)
    size = len(x_train)
    predictions = np.zeros((resamples, len(x_test)))
    for i in range(resamples):
        # витягуємо номери рядків з поверненням — частина обʼєктів повториться,
        # частина не потрапить зовсім, і саме це створює різницю між моделями
        chosen = rng.integers(0, size, size)
        predictions[i] = fit_polynomial(x_train[chosen], y_train[chosen], degree, x_test)
    return predictions


print(f"{'степінь':>8} {'дисперсія (світи)':>19} {'дисперсія (бутстреп)':>22} {'у скільки разів':>17}")
for degree in [1, 5, 9]:
    _, ideal_variance, _ = decompose(run_parallel_worlds(degree))
    bootstrap_variance = np.mean(
        bootstrap_predictions(x_train_one, y_train_one, degree).var(axis=0))
    print(f"{degree:>8} {ideal_variance:>19.4f} {bootstrap_variance:>22.4f} "
          f"{bootstrap_variance / ideal_variance:>17.1f}")

print("\nНапрямок бутстреп ловить правильно: дисперсія росте зі складністю.")
print("А от абсолютні числа він завищує, і тим сильніше, чим складніша модель.")
print("Причина: підвибірка з поверненням містить у середньому лише 63% різних")
print("обʼєктів. Гнучкому поліному цього критично мало — він хапається за дублікати,")
print("і його розкид роздувається. Тому бутстреп — індикатор, а не вимірювальний прилад.")

## 7. Дані б'ють лише по дисперсії

Лекція: $\text{Дисперсія} \approx C/n$, а зміщення від $n$ **не залежить взагалі**.
Перевіримо обидва твердження одразу, зафіксувавши складність.

In [ ]:
sample_sizes = np.array([40, 80, 160, 320, 640])
bias_by_size = np.zeros(len(sample_sizes))
variance_by_size = np.zeros(len(sample_sizes))

for i, size in enumerate(sample_sizes):
    predictions = run_parallel_worlds(degree=7, size=size, worlds=200)
    bias_by_size[i], variance_by_size[i], _ = decompose(predictions)

print(f"{'n':>6} {'зміщення²':>11} {'дисперсія':>11} {'дисперсія × n':>15}")
for i, size in enumerate(sample_sizes):
    print(f"{size:>6} {bias_by_size[i]:>11.5f} {variance_by_size[i]:>11.5f} "
          f"{variance_by_size[i] * size:>15.3f}")

print("\nЗміщення² стоїть на місці — скільки даних не давай, поліном 7-го степеня")
print("лишиться поліномом 7-го степеня. Це і є сенс слова «систематична».")
print(f"\nА дисперсія × n майже не рухається: {variance_by_size[0] * sample_sizes[0]:.2f} → "
      f"{variance_by_size[-1] * sample_sizes[-1]:.2f}, тоді як сама n виросла в "
      f"{sample_sizes[-1] // sample_sizes[0]} разів.")
print("Тобто дисперсія падає приблизно як 1/n — трохи швидше, бо на малих вибірках")
print("до неї домішується чисельна нестійкість поліноміальної підгонки.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.4))

ax.loglog(sample_sizes, variance_by_size, "o-", color="teal", lw=2.5, label="дисперсія")
ax.loglog(sample_sizes, bias_by_size + 1e-6, "o-", color="crimson", lw=2.5,
          label="зміщення² (+1e-6, щоб було видно на лог. шкалі)")
# еталонний нахил 1/n, прив'язаний до першої точки
reference = variance_by_size[0] * sample_sizes[0] / sample_sizes
ax.loglog(sample_sizes, reference, ls="--", color="grey", lw=2, label="еталон 1/n")

ax.set_xlabel("розмір навчальної вибірки n"); ax.set_ylabel("значення (лог. шкала)")
ax.set_title("Більше даних — менша дисперсія. Зміщення не рухається")
ax.legend(); ax.grid(alpha=.25, which="both")
plt.tight_layout(); plt.show()

## 8. Регуляризація: свідомий обмін

Найцікавіше. Візьмемо **навмисно завелику** модель — поліном 11-го степеня на
40 точках — і додамо штраф за величину коефіцієнтів:

$$L = \sum (y - \hat{y})^2 + \lambda \sum_{j \ge 1} \beta_j^2$$

Вільний член не штрафуємо: зсувати всю криву вгору-вниз — це не «зайва
гнучкість», і карати за це немає за що.

Питання одне: чи виграш у дисперсії перевищить програш у зміщенні?

In [ ]:
def run_worlds_with_ridge(lam, degree=11, worlds=N_WORLDS, seed=0):
    """Ті самі паралельні світи, але з L2-штрафом. Повертає матрицю прогнозів."""
    rng = np.random.default_rng(seed)
    penalty = np.eye(degree + 1)
    penalty[0, 0] = 0                      # вільний член не штрафуємо
    design_test = legendre.legvander(x_test, degree)

    predictions = np.zeros((worlds, len(x_test)))
    for world in range(worlds):
        x_train, y_train = draw_training_sample(rng)
        design = legendre.legvander(x_train, degree)
        # нормальне рівняння зі штрафом: (XᵀX + λI)β = Xᵀy
        coefficients = np.linalg.solve(design.T @ design + lam * penalty, design.T @ y_train)
        predictions[world] = design_test @ coefficients
    return predictions


lambdas = np.array([0.0, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0])
bias_by_lambda = np.zeros(len(lambdas))
variance_by_lambda = np.zeros(len(lambdas))

for i, lam in enumerate(lambdas):
    bias_by_lambda[i], variance_by_lambda[i], _ = decompose(run_worlds_with_ridge(lam))

total_by_lambda = bias_by_lambda + variance_by_lambda + NOISE_SIGMA ** 2
best_lambda = lambdas[total_by_lambda.argmin()]

print(f"{'λ':>10} {'зміщення²':>11} {'дисперсія':>11} {'разом':>9}")
for i, lam in enumerate(lambdas):
    mark = "  ← найкраще" if lam == best_lambda else ""
    print(f"{lam:>10} {bias_by_lambda[i]:>11.4f} {variance_by_lambda[i]:>11.4f} "
          f"{total_by_lambda[i]:>9.4f}{mark}")

print(f"\nБез штрафу помилка {total_by_lambda[0]:.4f}, з найкращим λ — {total_by_lambda.min():.4f}.")
print(f"Виграш у {total_by_lambda[0] / total_by_lambda.min():.1f} раза — і це та сама модель,")
print("того самого 11-го степеня. Ми лише заборонили їй користуватися всією гнучкістю.")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.4))

# нуль на логарифмічній осі не намалюєш, тому λ=0 показуємо окремою рискою
positive = lambdas > 0
ax.plot(lambdas[positive], bias_by_lambda[positive], "o-", color="crimson", lw=2.5,
        label="зміщення² — росте")
ax.plot(lambdas[positive], variance_by_lambda[positive], "o-", color="teal", lw=2.5,
        label="дисперсія — падає")
ax.plot(lambdas[positive], total_by_lambda[positive], "o-", color="black", lw=2.5,
        label="разом")
ax.axhline(total_by_lambda[0], ls=":", color="grey", label="разом при λ = 0")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("λ (лог. шкала)"); ax.set_ylabel("внесок у помилку (лог. шкала)")
ax.set_title("λ обмінює дисперсію на зміщення — і сума має мінімум")
ax.legend(); ax.grid(alpha=.25, which="both")
plt.tight_layout(); plt.show()

## 9. Ті самі криві готовими інструментами

Усе, що ми рахували руками, у `scikit-learn` уже є двома функціями. Писати руками
було потрібно, щоб зрозуміти, **що саме** вони рахують, — тепер можна користуватися.

- `validation_curve` — помилка від **складності** (наша U-подібна крива);
- `learning_curve` — помилка від **розміру вибірки** (та сама крива навчання,
  яку лекція радить будувати перед тим, як замовляти нові дані).

Обидві всередині роблять крос-валідацію, тому дають чесніші числа, ніж одне
відкладене розбиття. Візьмемо трохи більшу вибірку — 150 точок, щоб кривій
навчання було куди рости.

In [ ]:
from sklearn.model_selection import validation_curve, learning_curve, KFold
from sklearn.preprocessing import StandardScaler

bigger_rng = np.random.default_rng(11)
x_big, y_big = draw_training_sample(bigger_rng, size=150)
x_big_column = x_big.reshape(-1, 1)          # sklearn чекає матрицю, а не вектор

# у пайплайні степінь стоїть окремим кроком — саме його ми й будемо крутити
polynomial_model = make_pipeline(PolynomialFeatures(), StandardScaler(), LinearRegression())
folds = KFold(n_splits=5, shuffle=True, random_state=0)

degree_range = np.arange(1, 15)
train_scores, test_scores = validation_curve(
    polynomial_model, x_big_column, y_big,
    param_name="polynomialfeatures__degree", param_range=degree_range,
    cv=folds, scoring="neg_mean_squared_error")

# sklearn повертає «чим більше, тим краще», тому міняємо знак назад на MSE
train_mse = -train_scores.mean(axis=1)
test_mse = -test_scores.mean(axis=1)
best_by_library = degree_range[test_mse.argmin()]

print(f"{'степінь':>8} {'train':>9} {'test':>9}")
for i, degree in enumerate(degree_range):
    mark = "  ← мінімум" if degree == best_by_library else ""
    print(f"{degree:>8} {train_mse[i]:>9.4f} {test_mse[i]:>9.4f}{mark}")

print(f"\nvalidation_curve на ОДНІЙ вибірці каже: степінь {best_by_library}.")
print(f"Наш чесний експеримент із {N_WORLDS} паралельними світами казав: степінь {best_degree}.")
print("Два зовсім різні шляхи привели до однієї відповіді — це і є найкраща перевірка.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

axes[0].plot(degree_range, train_mse, "o-", color="crimson", lw=2.5, label="train")
axes[0].plot(degree_range, test_mse, "o-", color="teal", lw=2.5, label="test (CV)")
axes[0].axvline(best_by_library, ls="--", color="grey", label=f"мінімум: {best_by_library}")
axes[0].set_xlabel("степінь полінома"); axes[0].set_title("validation_curve: помилка від складності")

# крива навчання для замалої і для доречної моделі
for degree, colour in [(1, "crimson"), (5, "teal")]:
    sizes, _, holdout_scores = learning_curve(
        make_pipeline(PolynomialFeatures(degree), StandardScaler(), LinearRegression()),
        x_big_column, y_big, train_sizes=np.linspace(0.2, 1.0, 7),
        cv=folds, scoring="neg_mean_squared_error")
    axes[1].plot(sizes, -holdout_scores.mean(axis=1), "o-", color=colour, lw=2.5,
                 label=f"степінь {degree}")

axes[1].axhline(NOISE_SIGMA ** 2, ls=":", color="grey", label="підлога σ²")
axes[1].set_xlabel("розмір навчальної частини"); axes[1].set_title("learning_curve: помилка від кількості даних")

for ax in axes:
    ax.set_ylabel("MSE"); ax.legend(); ax.grid(alpha=.25)
plt.tight_layout(); plt.show()

print("Читаємо праву картинку так, як радить лекція:")
print("  степінь 1 — крива вийшла на плато високо над підлогою. Дані вже не")
print("              допоможуть, проблема в моделі. Замовляти розмітку — марно.")
print("  степінь 5 — крива ще спадає й тисне до підлоги σ². Ось тут нові дані")
print("              справді куплять тобі якість.")

---

## 💻 Завдання

### 🟢 Рівень 1 — разом
1. Зміни `NOISE_SIGMA` з `0.3` на `0.6` і перезапусти розділ 4. Який доданок
   змінився, а які лишились на місці? Чи зсунувся оптимальний степінь — і в який бік?
2. Постав `SAMPLE_SIZE = 15` і подивись на таблицю розкладу. На якому степені
   тепер починається катастрофа? Поясни зв'язок із кількістю параметрів моделі.

### 🟡 Рівень 2 — самостійно
1. Заміни поліном на **k-NN** (`sklearn.neighbors.KNeighborsRegressor`) і побудуй
   ту саму таблицю зміщення²/дисперсії, але по $k$ від 1 до 25. Функція
   `run_parallel_worlds` майже не зміниться — треба лише інакше навчати модель.
2. Побудуй **криву навчання**: помилка від розміру вибірки при фіксованій
   складності, для степенів 1 і 9 на одному графіку.

**Зроблено, якщо:** у k-NN зміщення й дисперсія поводяться **дзеркально** до
полінома (мале $k$ = велика дисперсія), і ти написав(ла) одним реченням, чому саме
так. А на кривій навчання видно, що для степеня 1 вона виходить на плато високо,
а для степеня 9 продовжує падати.

### 🔴 Рівень 3 — виклик
1. Візьми `sklearn.ensemble.RandomForestRegressor` і покажи числом те, про що
   говорить зірочка в таблиці розділу 10 лекції: при зростанні `n_estimators`
   від 1 до 200 **дисперсія падає, а зміщення² майже не рухається**.
2. Порівняй це з `GradientBoostingRegressor`, у якого зростання `n_estimators`
   спершу знижує зміщення, а потім розганяє дисперсію.

**Зроблено, якщо:** є два графіки «зміщення² і дисперсія від кількості дерев» —
для лісу й для бустингу — і по них видно, що ансамблі атакують **різні** доданки.